## Face-to-Voice Inference

### Overview
This notebook demonstrates inference using the trained Face-to-Voice models.  
Speech audio is generated from facial images by predicting Voice Style Vectors and feeding them into StyleTTS2.

### Workflow
1.  **Setup**: Import libraries, define constants, load the dataset and trained models.
2.  **Preprocessing**: Split the dataset and prepare input vectors by normalizing face embeddings and encoding age categories.
3.  **Inference**: Predict voice embeddings from face vectors and restore the scale.
4.  **Audio Synthesis**: Generate speech using StyleTTS2 with predicted embeddings.
5.  **Evaluation**: Compare generated audio with ground truth by listening to samples.

### 1. Setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import contextlib
import warnings
import torch

# Machine Learning & Deep Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import tensorflow as tf
from tensorflow.keras.models import load_model
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# AI Models
from deepface import DeepFace
from styletts2.tts import StyleTTS2

from IPython.display import Audio, display

# Settings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# --- Configuration ---
DATASET_ROOT = '../data'
MODELS_DIR = '../models'

# Paths
DATASET_PATH = os.path.normpath(os.path.join(DATASET_ROOT, 'face_to_voice_dataset.pkl'))
MALE_MODEL_PATH = os.path.normpath(os.path.join(MODELS_DIR, 'model_m.h5'))
FEMALE_MODEL_PATH = os.path.normpath(os.path.join(MODELS_DIR, 'model_f.h5'))

# Constants
SEED = 42
MALE_VOICE_SCALE = 3.38  # Mean Male L2 norm of StyleTTS2 embeddings
FEMALE_VOICE_SCALE = 3.39  # Mean Female L2 norm of StyleTTS2 embeddings
SPEAKING_TEXT = 'Hello. This is a voice generated from the face image.'

# --- Reproducibility ---
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# Set seed
seed_everything(SEED)

In [ ]:
# --- Load Dataset ---
if os.path.exists(DATASET_PATH):
    df_org = pd.read_pickle(DATASET_PATH)

    # Create a working copy
    df = df_org.copy()

    print(f'Dataset loaded. Shape: {df.shape}')
else:
    raise FileNotFoundError(f'File not found: {DATASET_PATH}')

# --- Load Models ---
with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
    tts = StyleTTS2()

male_model = load_model(MALE_MODEL_PATH, compile=False)
female_model = load_model(FEMALE_MODEL_PATH, compile=False)

### 2. Preprocessing

In [ ]:
# Train/Test Split for Male
df_male = df[df['Gender'] == 'm'].copy()
df_train_m, df_test_m = train_test_split(
        df_male, 
        test_size=0.1, 
        random_state=SEED, 
        shuffle=True
    )

# Train/Test Split for Female
df_female = df[df['Gender'] == 'f'].copy()
df_train_f, df_test_f = train_test_split(
        df_female, 
        test_size=0.1, 
        random_state=SEED, 
        shuffle=True
    )

# Preprocessing
def prepare_input_vector(row):
    # L2 Normalization
    face_vec = np.array(row['face_vector'], dtype=np.float32).reshape(1, -1)
    face_vec_norm = normalize(face_vec, norm='l2')
    
    # Categorize Age
    age_bins = [0, 20, 30, 40, 50, 60, np.inf]
    age_labels = ['0-19', '20-29', '30-39', '40-49', '50-59', '60+']
    age_category = pd.cut([row['age']], bins=age_bins, labels=age_labels, right=False)[0]

    # One-Hot Encoding
    age_idx = age_labels.index(age_category)
    age = np.zeros((1, 6), dtype=np.float32)
    age[0, age_idx] = 1.0
    
    # Concatenate Inputs
    input_vector = np.hstack([face_vec_norm, age])
    
    return input_vector

### 3. Inference (Face → Voice)

In [ ]:
def predict_voice_embedding(input_vector, gender):
    if gender == 'm':
        pred = male_model.predict(input_vector, verbose=0)
        # Scale Restoration
        pred_scaled = pred * MALE_VOICE_SCALE
    else:
        pred = female_model.predict(input_vector, verbose=0)
        # Scale Restoration
        pred_scaled = pred * FEMALE_VOICE_SCALE    
    
    # Numpy Array -> PyTorch Tensor
    speaker_embedding = torch.tensor(pred_scaled, dtype=torch.float32).to(device)
    
    return speaker_embedding

### 4. Audio Synthesis

In [ ]:
def run_face_to_voice_demo(gender):
    selected_sample = None

    if gender == 'm':
        selected_sample = df_test_m.sample(1).iloc[0].copy()
    else:
        selected_sample = df_test_f.sample(1).iloc[0].copy()

    vox_id = selected_sample['VoxCeleb2 ID']

    print(f'Processing Speaker: {vox_id}')
    print(f"Demographics: Age {selected_sample['age']}, Sex {selected_sample['Gender']}")

    # Input Processing
    input_vec = prepare_input_vector(selected_sample)

    # Inference
    pred_embedding = predict_voice_embedding(input_vec, gender)
    
    # Generate with ground truth embedding
    try:
        gt_embedding = torch.tensor(selected_sample['style_vectors'], dtype=torch.float32).reshape(1, -1).to(device)
        wav_recon = tts.inference(
            text=SPEAKING_TEXT,
            ref_s=gt_embedding
        )

        print('\n🔊 Generated voice (ground truth):')
        display(Audio(wav_recon, rate=24000))
            
    except Exception as e:
        print(f'❌ Generation Failed: {e}')

    # Generate with predicted embedding
    try:
        wav_gen = tts.inference(
            text=SPEAKING_TEXT,
            ref_s=pred_embedding
        )

        print('\n🔊 Generated voice (predicted):')
        display(Audio(wav_gen, rate=24000))
        
    except Exception as e:
        print(f'❌ Generation Failed: {e}')

### 5. Evaluation

In [ ]:
for i in range(1):
    run_face_to_voice_demo('m')  # male sample
    run_face_to_voice_demo('f')  # female sample

**Results**  
The model successfully generates voice from face images with the following characteristics:

- **Gender & Age**: Voices are clearly distinguished by gender, and age-related variation is perceivable.
- **Face-Voice Consistency**: The predicted voice feels natural and plausible when viewed alongside the input face image.
- **Pitch**: The predicted voices tend to be slightly higher in pitch compared to the ground truth.